# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MbaliThusi/Mbali-Flyrank-intern-rep./blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [ ]:
import pandas as pd, numpy as np
from google.colab import files
uploaded = files.upload()
df = pd.read_csv(list(uploaded.keys())[0])

base = df[df["clicks_prev_30d"] > 0].copy()
y = (base["clicks_last_30d"] < base["clicks_prev_30d"]).astype(int)

cat_cols = ["content_type", "main_intent", "competition_level", "provider_used", "model_used"]
num_cols = ["search_volume", "competition", "cpc", "word_count", "char_count",
            "content_age_days", "days_since_last_update",
            "impressions_prev_30d", "clicks_prev_30d", "sessions_prev_30d"]
X = base[num_cols + cat_cols].copy()
X["chars_per_word"] = base["char_count"] / base["word_count"].replace(0, np.nan)
X["prev_ctr"] = base["clicks_prev_30d"] / base["impressions_prev_30d"].replace(0, np.nan)
X["log_clicks_prev"] = np.log1p(base["clicks_prev_30d"])
X["log_impr_prev"] = np.log1p(base["impressions_prev_30d"])
X = X.replace([np.inf, -np.inf], np.nan)
for c in list(X.select_dtypes("number").columns):
    if X[c].isna().any():
        X[c + "_was_missing"] = X[c].isna().astype(int)
        X[c] = X[c].fillna(X[c].median())
X[cat_cols] = X[cat_cols].fillna("unknown")
X = pd.get_dummies(X, columns=cat_cols, drop_first=True).astype(float)
groups = base["client_id"]
print(X.shape, "| positive rate:", y.mean().round(3))

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [ ]:
print("rows:", len(X), "| declined:", y.mean().round(3), "| clients:", groups.nunique())

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [ ]:
from sklearn.model_selection import GroupKFold

gkf = GroupKFold(n_splits=5)
splits = list(gkf.split(X, y, groups))
for i, (tr, te) in enumerate(splits):
    print(f"fold {i}: train={len(tr)}, test={len(te)}, test clients={groups.iloc[te].nunique()}, "
          f"test decline rate={y.iloc[te].mean():.3f}")
assert all(set(groups.iloc[tr]).isdisjoint(groups.iloc[te]) for tr, te in splits)
print("no client appears in both train and test")

## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score, average_precision_score

def prec_at_top10(yt, s):
    k = int(len(s) * 0.10)
    return np.asarray(yt)[np.argsort(-np.asarray(s))[:k]].mean()

models = {
    "logistic": make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
    "boosting": HistGradientBoostingClassifier(max_depth=4, learning_rate=0.05, max_iter=200, random_state=0),
}
oof = {n: np.zeros(len(X)) for n in list(models) + ["baseline_stale"]}
rows = []

for i, (tr, te) in enumerate(splits):
    for n, m in models.items():
        m.fit(X.iloc[tr], y.iloc[tr])
        oof[n][te] = m.predict_proba(X.iloc[te])[:, 1]
    oof["baseline_stale"][te] = base["days_since_last_update"].iloc[te].fillna(0).values
    for n in oof:
        s, yt = oof[n][te], y.iloc[te]
        rows.append([n, i, roc_auc_score(yt, s), average_precision_score(yt, s), prec_at_top10(yt, s)])

res = pd.DataFrame(rows, columns=["model", "fold", "AUC", "AvgPrecision", "Prec@top10%"])
table = res.groupby("model")[["AUC", "AvgPrecision", "Prec@top10%"]].agg(["mean", "std"]).round(3)
print(table)
print("\nrandom guess: AUC 0.5, precision =", y.mean().round(3))

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [11]:
from sklearn.inspection import permutation_importance
from sklearn.metrics import roc_auc_score

# What does it lean on? (held-out fold 0, boosting)
tr, te = splits[0]
m = models["boosting"].fit(X.iloc[tr], y.iloc[tr])
pi = permutation_importance(m, X.iloc[te], y.iloc[te], scoring="roc_auc", n_repeats=3, random_state=0)
imp = pd.Series(pi.importances_mean, index=X.columns).sort_values(ascending=False)
print("top features by AUC drop when shuffled:\n", imp.head(8).round(4))

# Where is it wrong?
e = base[["client_id", "content_type", "clicks_prev_30d"]].copy()
e["content_type"] = e["content_type"].fillna("unknown").astype(str)
e["y"] = y.values
e["p"] = oof["boosting"]
e["wrong"] = ((e["p"] > 0.5).astype(int) != e["y"]).astype(int)

print("\nerror rate by traffic level:")
traffic = pd.qcut(e["clicks_prev_30d"], 4, duplicates="drop")
print(e.groupby(traffic, observed=True)["wrong"].mean().round(3))

print("\nerror rate by content_type:")
ct = e.groupby("content_type")["wrong"].agg(["mean", "size"]).round(3)
print(ct.sort_values("mean", ascending=False).head(6))

# Per-client AUC (plain loop, no groupby.apply)
vals = []
for cid, g in e.groupby("client_id"):
    if g["y"].nunique() == 2 and len(g) >= 50:
        vals.append(roc_auc_score(g["y"], g["p"]))
pc = pd.Series(vals)
print("\nper-client AUC: min", round(pc.min(), 3), "| median", round(pc.median(), 3),
      "| max", round(pc.max(), 3), "| clients:", len(pc))

top features by AUC drop when shuffled:
 prev_ctr                  0.1057
clicks_prev_30d           0.0308
days_since_last_update    0.0067
impressions_prev_30d      0.0024
search_volume             0.0018
sessions_prev_30d         0.0012
char_count                0.0006
main_intent_unknown       0.0001
dtype: float64

error rate by traffic level:
clicks_prev_30d
(0.999, 3.0]      0.355
(3.0, 10.0]       0.444
(10.0, 1627.0]    0.351
Name: wrong, dtype: float64

error rate by content_type:
                     mean   size
content_type                    
keyword article     0.379  11476
feedly article      0.218    243
comparison article  0.125     40

per-client AUC: min 0.53 | median 0.653 | max 0.798 | clients: 17


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.